# Telco Customer Churn Analysis

## Problem Definition
Customer churn represents one of the most significant challenges for telecommunication companies. Acquiring new customers is often substantially more expensive than retaining existing ones.

Our objective is to analyze the historical customer data provided in `Telco_customer_churn.xlsx` to:
1. **Understand Key Drivers:** Identify the primary factors that influence a customer's decision to leave (churn) the service.
2. **Predict Churn Risk:** Develop a predictive model capable of identifying high-risk customers before they churn, enabling proactive retention strategies.
3. **Provide Actionable Insights:** Recommend targeted marketing, pricing, or customer service interventions based on the analysis to improve overall retention rates.

In [20]:
import pandas as pd

# Load the Excel dataset
df = pd.read_excel('Telco_customer_churn.xlsx')

# Display the first few rows
display(df.head())

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


## Step 1: Initial Data Inspection
Let's check the structural summary of the DataFrame, including the names of the columns, their data types, and the count of non-null values to identify missing data.

In [21]:
# Show the summary of the dataset
print("Dataset Shape:", df.shape)
print("\n--- Dataset Info ---")
df.info()

# Check for missing values explicitly
print("\n--- Missing Values Count ---")
missing_vals = df.isnull().sum()
display(missing_vals[missing_vals > 0])

Dataset Shape: (7043, 33)

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 

,0
Churn Reason,5174


## Step 2: Data Cleaning and Type Conversion
We need to convert `Total Charges` to a numeric float type. Often, blank spaces or empty strings in this column represent new customers with 0 tenure, which causes pandas to treat the column as an object type.

In [22]:
# Identify blank spaces in Total Charges
blank_total_charges = df[df['Total Charges'].astype(str).str.strip() == '']
print(f"Number of rows with blank 'Total Charges': {len(blank_total_charges)}")

# Convert Total Charges to numeric, forcing errors to NaN
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

# Fill NaN values in Total Charges with 0 (since these are customers with 0 tenure months)
df['Total Charges'] = df['Total Charges'].fillna(0.0)

# Let's verify the updated data types
print("\nUpdated 'Total Charges' Dtype:", df['Total Charges'].dtype)

Number of rows with blank 'Total Charges': 11

Updated 'Total Charges' Dtype: float64


## Step 3: Target Variable Distribution & Descriptive Statistics
Let's evaluate the class balance of the target variable (`Churn Label`) and get a statistical summary of the key continuous numerical variables (`Tenure Months`, `Monthly Charges`, `Total Charges`, `CLTV`).

In [23]:
import plotly.express as px

# Calculate target distribution percentages
churn_pct = df['Churn Label'].value_counts(normalize=True) * 100
print("Churn Rates:")
print(churn_pct.round(2))

# Create interactive bar chart
fig = px.bar(
    x=churn_pct.index,
    y=churn_pct.values,
    labels={'x': 'Churn Status', 'y': 'Percentage (%)'},
    title='Customer Churn Distribution (%)',
    color=churn_pct.index,
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig.show()

Churn Rates:
Churn Label
No     73.46
Yes    26.54
Name: proportion, dtype: float64


## Step 4: Numerical Feature Distributions vs. Churn
Let's visualize how `Tenure Months`, `Monthly Charges`, and `CLTV` relate to whether a customer churns. Box plots will help us see if there are noticeable shifts in median values or distributions.

In [24]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Subplots for numerical features vs churn
fig = make_subplots(rows=1, cols=3, subplot_titles=('Tenure Months vs. Churn', 'Monthly Charges vs. Churn', 'CLTV vs. Churn'))

for i, col in enumerate(['Tenure Months', 'Monthly Charges', 'CLTV'], 1):
    for status in ['Yes', 'No']:
        fig.add_trace(
            go.Box(y=df[df['Churn Label'] == status][col], name=f"{col} ({status})", boxpoints='outliers'),
            row=1, col=i
        )

fig.update_layout(title_text="Numerical Features by Churn Status", showlegend=False, height=500)
fig.show()

## Step 5: Categorical Features vs. Churn
Let's analyze how categorical features like `Contract` type and `Internet Service` type relate to churn. We will use stacked bar plots to compare the percentage of churn within each category.

In [25]:
# Contract type distribution vs churn
fig1 = px.histogram(
    df, x='Contract', color='Churn Label', barnorm='percent',
    title='Churn Rate by Contract Type',
    color_discrete_sequence=['#e28743', '#abdbe3']
)
fig1.update_layout(yaxis_title="Percentage (%)")
fig1.show()

# Internet Service type distribution vs churn
fig2 = px.histogram(
    df, x='Internet Service', color='Churn Label', barnorm='percent',
    title='Churn Rate by Internet Service Type',
    color_discrete_sequence=['#e28743', '#abdbe3']
)
fig2.update_layout(yaxis_title="Percentage (%)")
fig2.show()

## Step 6: Correlation Analysis
Let's calculate the Pearson correlation coefficients among our numerical variables (including the binary target `Churn Value`) to see how strongly these features are correlated with each other and with the churn outcome.

In [26]:
# Heatmap of numerical feature correlations
numeric_cols = ['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV', 'Churn Score', 'Churn Value']
corr_matrix = df[numeric_cols].corr()

fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Correlation Heatmap'
)
fig.show()

## Feature Engineering
In this step, we will drop columns that don't add predictive value, encode binary text columns to `0`/`1`, and apply one-hot encoding to multi-class categorical variables.

In [27]:
# 1. Drop redundant or non-predictive columns
columns_to_drop = ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Churn Label', 'Churn Reason']
df_ml = df.drop(columns=columns_to_drop, errors='ignore')

# 2. Binary Encoding for Yes/No columns & Gender
binary_cols = ['Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Paperless Billing']
for col in binary_cols:
    df_ml[col] = df_ml[col].map({'Yes': 1, 'No': 0})

df_ml['Gender'] = df_ml['Gender'].map({'Female': 1, 'Male': 0})

# 3. One-Hot Encoding for remaining categorical variables
categorical_cols = ['Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup',
                    'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies',
                    'Contract', 'Payment Method']

df_ml = pd.get_dummies(df_ml, columns=categorical_cols, drop_first=True)

# Display the prepared dataset
print("Features shape after encoding:", df_ml.shape)
display(df_ml.head())

Features shape after encoding: (7043, 33)


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Paperless Billing,Monthly Charges,Total Charges,Churn Value,...,Tech Support_Yes,Streaming TV_No internet service,Streaming TV_Yes,Streaming Movies_No internet service,Streaming Movies_Yes,Contract_One year,Contract_Two year,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,0,0,0,0,2,1,1,53.85,108.15,1,...,False,False,False,False,False,False,False,False,False,True
1,1,0,0,1,2,1,1,70.70,151.65,1,...,False,False,False,False,False,False,False,False,True,False
2,1,0,0,1,8,1,1,99.65,820.50,1,...,False,False,True,False,True,False,False,False,True,False
3,1,0,1,1,28,1,1,104.80,3046.05,1,...,True,False,True,False,True,False,False,False,True,False
4,0,0,0,1,49,1,1,103.70,5036.30,1,...,False,False,True,False,True,False,False,False,False,False


## Step 7: Train-Test Split and Feature Scaling
In this step, we split our data into training and test sets. We drop `Churn Score` to avoid data leakage since it is calculated after churn. We then scale continuous numerical features using `StandardScaler` to ensure our distance-based or regularized models behave optimally.

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features and target
# We drop Churn Value (target) and Churn Score (potential leakage/post-facto score)
X = df_ml.drop(columns=['Churn Value', 'Churn Score'], errors='ignore')
y = df_ml['Churn Value']

# Split into training and testing sets (80% train, 20% test) with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale continuous numerical features
scaler = StandardScaler()
num_features_to_scale = ['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']

# Fit on training data and transform both train and test sets
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_features_to_scale] = scaler.fit_transform(X_train[num_features_to_scale])
X_test_scaled[num_features_to_scale] = scaler.transform(X_test[num_features_to_scale])

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")
print("Target distribution in y_train:\n", y_train.value_counts(normalize=True))

Training set shape: (5634, 31)
Test set shape: (1409, 31)
Target distribution in y_train:
 Churn Value
0    0.734647
1    0.265353
Name: proportion, dtype: float64


## Step 8: Model Training and Evaluation
Here, we train a baseline **Logistic Regression** model and a **Random Forest Classifier**. We will compute key classification metrics (Precision, Recall, F1-Score) and plot the ROC Curve to determine which model performs better at identifying high-risk churn candidates.

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
}

fig = go.Figure()
fig.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=0, y1=1)

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    print(f"=== {name} Classification Report ===")
    print(classification_report(y_test, y_pred))

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f"{name} (AUC = {auc_score:.3f})", mode='lines'))

fig.update_layout(title='ROC Curve', xaxis_title='False Positive Rate', yaxis_title='True Positive Rate', width=700, height=600)
fig.show()

=== Logistic Regression Classification Report ===
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.65      0.57      0.61       374

    accuracy                           0.80      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.80      0.80      1409

=== Random Forest Classification Report ===
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1035
           1       0.66      0.50      0.57       374

    accuracy                           0.80      1409
   macro avg       0.75      0.70      0.72      1409
weighted avg       0.79      0.80      0.79      1409



## Step 9: Feature Importance & Churn Drivers Analysis
To extract actionable business insights, we analyze the coefficients of our best-performing baseline model (Logistic Regression). Positive coefficients point to features that increase the probability of churn, while negative coefficients indicate features that help retain customers.

In [30]:
# Extract Logistic Regression coefficients and sort
log_reg_model = models['Logistic Regression']
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_reg_model.coef_[0]
})
feature_importance_df['Abs_Coefficient'] = feature_importance_df['Coefficient'].abs()
feature_importance_df = feature_importance_df.sort_values(by='Abs_Coefficient', ascending=False).head(15)

# Create interactive horizontal bar chart
fig = px.bar(
    feature_importance_df,
    x='Coefficient',
    y='Feature',
    orientation='h',
    title='Top 15 Churn Drivers',
    color='Coefficient',
    color_continuous_scale='curl'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## Step 10: Hyperparameter Tuning & Advanced Evaluation
To achieve peak model performance, we perform a hyperparameter grid search on the `Random Forest` model using 3-fold cross-validation. We then evaluate and compare the prediction errors using an interactive confusion matrix.

In [31]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix

# Define parameter grid
param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [8, 12, None],
    'min_samples_split': [2, 5]
}

# Optimize Random Forest using Grid Search
rf = RandomForestClassifier(random_state=42, class_weight='balanced')
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

best_rf = grid_search.best_estimator_
print("Optimized Parameters:", grid_search.best_params_)

# Predict using optimized model
y_pred_rf = best_rf.predict(X_test_scaled)
y_pred_lr = models['Logistic Regression'].predict(X_test_scaled)

# Calculate confusion matrices
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_rf = confusion_matrix(y_test, y_pred_rf)

# Visualize side-by-side interactive confusion matrices
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Logistic Regression Confusion Matrix', 'Optimized Random Forest Confusion Matrix')
)

# Add Logistic Regression CM
fig.add_trace(
    go.Heatmap(
        z=cm_lr, x=['Predicted No Churn', 'Predicted Churn'], y=['Actual No Churn', 'Actual Churn'],
        coloraxis='coloraxis', text=cm_lr, texttemplate="%{text}"
    ), row=1, col=1
)

# Add Random Forest CM
fig.add_trace(
    go.Heatmap(
        z=cm_rf, x=['Predicted No Churn', 'Predicted Churn'], y=['Actual No Churn', 'Actual Churn'],
        coloraxis='coloraxis', text=cm_rf, texttemplate="%{text}"
    ), row=1, col=2
)

fig.update_layout(coloraxis=dict(colorscale='Blues'), showlegend=False, height=500)
fig.show()

Optimized Parameters: {'max_depth': 8, 'min_samples_split': 2, 'n_estimators': 100}


## Step 11: Export Predictions to CSV
In this final step, we use our trained model to make predictions and calculate churn probability scores for all customers. We save these predictions along with the unique `CustomerID` to a CSV file for targeted marketing campaigns.

In [32]:
# Prepare the full feature set matching training format
X_full = df_ml.drop(columns=['Churn Value', 'Churn Score'], errors='ignore')

# Scale full dataset numerical columns
X_full_scaled = X_full.copy()
X_full_scaled[num_features_to_scale] = scaler.transform(X_full[num_features_to_scale])

# Predict probabilities and classes using optimized Random Forest model
full_probabilities = best_rf.predict_proba(X_full_scaled)[:, 1]
full_predictions = best_rf.predict(X_full_scaled)

# Create export DataFrame
export_df = pd.DataFrame({
    'CustomerID': df['CustomerID'],
    'Actual_Churn': df['Churn Value'],
    'Predicted_Churn_Risk_Probability': full_probabilities.round(4),
    'Predicted_Churn_Label': full_predictions
})

# Save to CSV
export_filename = 'churn_predictions.csv'
export_df.to_csv(export_filename, index=False)
print(f"Successfully exported {len(export_df)} predictions to '{export_filename}'!")
display(export_df.head())

Successfully exported 7043 predictions to 'churn_predictions.csv'!


,CustomerID,Actual_Churn,Predicted_Churn_Risk_Probability,Predicted_Churn_Label
0,3668-QPYBK,1,0.6717,1
1,9237-HQITU,1,0.7647,1
2,9305-CDSKC,1,0.7463,1
3,7892-POOKP,1,0.4956,0
4,0280-XJGEX,1,0.3809,0


### Step 12: Interactive Dashboard & Static Export
We will now construct a comprehensive, interactive business dashboard using `plotly` that visually organizes key insights from our churn analysis. We will also install `kaleido` to enable high-quality static image exports directly from Plotly.

In [36]:
!pip install -q "kaleido==0.2.1"

In [43]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# 1. Compute KPI values
total_customers = len(df)
overall_churn_rate = (df['Churn Value'].mean() * 100)

# 2. Extract Top 8 Drivers
top_drivers = feature_importance_df.head(8).sort_values(by='Coefficient', ascending=True)

# 3. Compute Contract splits
contract_churn = df.groupby(['Contract', 'Churn Label']).size().unstack(fill_value=0)
contract_churn_pct = contract_churn.div(contract_churn.sum(axis=1), axis=0) * 100

# Create a structured 3-row grid layout to completely isolate KPI Cards in Row 1
fig = make_subplots(
    rows=3, cols=2,
    specs=[
        [{"type": "domain"}, {"type": "domain"}],  # Row 1 is dedicated exclusively to KPI Cards
        [{"type": "xy"}, {"type": "xy"}],          # Row 2 contains Churn Drivers & Contract Type Charts
        [{"type": "xy"}, {"type": "xy"}]           # Row 3 contains Monthly Charges & Churn Score Density
    ],
    subplot_titles=(
        "", "", # Row 1 KPI cards have titles built-in
        "Top Predictor Coefficients (Logistic Regression)",
        "Churn Proportions by Contract Type",
        "Monthly Charges Distribution",
        "Churn Score Density by Churn Status"
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.15
)

# --- ROW 1: Dedicated KPI Cards (No overlaps) ---
fig.add_trace(
    go.Indicator(
        mode="number",
        value=total_customers,
        title={"text": "Total Customers Analysed", "font": {"size": 16, "color": "#333333"}}
    ), row=1, col=1
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=round(overall_churn_rate, 2),
        number={"suffix": "%"},
        title={"text": "Average Churn Rate", "font": {"size": 16, "color": "#333333"}}
    ), row=1, col=2
)

# --- ROW 2 (LEFT): Top Churn Drivers ---
fig.add_trace(
    go.Bar(
        x=top_drivers['Coefficient'],
        y=top_drivers['Feature'],
        orientation='h',
        marker=dict(color=top_drivers['Coefficient'], colorscale='RdYlBu_r'),
        showlegend=False
    ), row=2, col=1
)

# --- ROW 2 (RIGHT): Churn Rate by Contract ---
for label, col_color in zip(['No', 'Yes'], ['#abdbe3', '#e28743']):
    fig.add_trace(
        go.Bar(
            x=contract_churn_pct.index,
            y=contract_churn_pct[label],
            name=f"Churn: {label}",
            marker_color=col_color,
            showlegend=True
        ), row=2, col=2
    )

# --- ROW 3 (LEFT): Monthly Charges Dist ---
fig.add_trace(
    go.Box(
        y=df[df['Churn Label'] == 'Yes']['Monthly Charges'],
        name='Churned (Yes)',
        marker_color='#e28743',
        boxpoints='outliers'
    ), row=3, col=1
)
fig.add_trace(
    go.Box(
        y=df[df['Churn Label'] == 'No']['Monthly Charges'],
        name='Retained (No)',
        marker_color='#abdbe3',
        boxpoints='outliers'
    ), row=3, col=1
)

# --- ROW 3 (RIGHT): Churn Score Density ---
fig.add_trace(
    go.Histogram(
        x=df[df['Churn Label'] == 'Yes']['Churn Score'],
        name='Churned (Yes)',
        marker_color='#e28743',
        opacity=0.7,
        nbinsx=20,
        showlegend=False
    ), row=3, col=2
)
fig.add_trace(
    go.Histogram(
        x=df[df['Churn Label'] == 'No']['Churn Score'],
        name='Retained (No)',
        marker_color='#abdbe3',
        opacity=0.7,
        nbinsx=20,
        showlegend=False
    ), row=3, col=2
)

# --- Dashboard Layout Customization (Adjusted aspect ratio to be explicitly horizontal) ---
fig.update_layout(
    title=dict(
        text="<b>TELCO CUSTOMER CHURN ENTERPRISE EXECUTIVE DASHBOARD</b>",
        font=dict(size=18, color="#1a2530"),
        x=0.5,
        y=0.98
    ),
    barmode='stack',
    width=1600,   # Widescreen landscape width
    height=900,   # Balanced landscape height to enforce a horizontal layout
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='center', x=0.5),
    margin=dict(t=100, b=60, l=120, r=40)
)

# Adjust individual axes properly
fig.update_yaxes(categoryorder='total ascending', row=2, col=1)
fig.update_yaxes(title_text="Proportion %", row=2, col=2)
fig.update_yaxes(title_text="Monthly Charges ($)", row=3, col=1)
fig.update_yaxes(title_text="Frequency Count", row=3, col=2)
fig.update_xaxes(title_text="Churn Score", row=3, col=2)

# Display the perfectly spaced interactive dashboard
fig.show()

# Save clean widescreen landscape snapshot as a PNG file
try:
    from kaleido.scopes.plotly import PlotlyScope
    scope = PlotlyScope()
    with open("churn_executive_dashboard.png", "wb") as f:
        f.write(scope.transform(fig, format="png", width=1600, height=900, scale=2))
    print("✔ Horizontal widescreen (1600x900) Executive Dashboard saved successfully to 'churn_executive_dashboard.png'!")
except Exception as e:
    print(f"Export failed: {e}")

✔ Horizontal widescreen (1600x900) Executive Dashboard saved successfully to 'churn_executive_dashboard.png'!
